<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula05%20-%20gridsearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [84]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = KNeighborsClassifier()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9444444444444444


In [89]:
X_train_small, X_val, y_train_small, y_val = train_test_split(X_train_scaled, y_train, test_size=0.2)

k_values = [1, 3, 5, 7, 9, 11, 13, 15]
best_k = None
best_accuracy = 0

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_small, y_train_small)
    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_k = k

print("Best k:", best_k)
print("Best accuracy:", best_accuracy)

Best k: 3
Best accuracy: 1.0


In [117]:
from sklearn.model_selection import cross_validate, RepeatedStratifiedKFold

splitter = RepeatedStratifiedKFold(n_splits=5)

k_values = [1, 3, 5, 7, 9, 11, 13, 15]
best_k = None
best_accuracy = 0

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    cv_results = cross_validate(model, X_train_scaled, y_train, cv=splitter, scoring='accuracy')
    accuracy = cv_results['test_score'].mean()
    print("k =", k, "Accuracy:", accuracy)
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_k = k

print("Best k:", best_k)
print("Best accuracy:", best_accuracy)

k = 1 Accuracy: 0.9520935960591135
k = 3 Accuracy: 0.9562807881773401
k = 5 Accuracy: 0.9543103448275864
k = 7 Accuracy: 0.9584729064039409
k = 9 Accuracy: 0.9555911330049263
k = 11 Accuracy: 0.9577339901477834
k = 13 Accuracy: 0.9655911330049263
k = 15 Accuracy: 0.9612807881773401
Best k: 13
Best accuracy: 0.9655911330049263


In [135]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

splitter = StratifiedKFold(n_splits=5, shuffle=True)

k_values = [1, 3, 5, 7, 9, 11, 13, 15]
best_k = None
best_accuracy = 0

for k in k_values:
    model = Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(k))])
    cv_results = cross_validate(model, X_train, y_train, cv=splitter, scoring='accuracy')
    accuracy = cv_results['test_score'].mean()
    print("k =", k, "Accuracy:", accuracy)
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_k = k

print("Best k:", best_k)
print("Best accuracy:", best_accuracy)

k = 1 Accuracy: 0.9576354679802955
k = 3 Accuracy: 0.9504926108374384
k = 5 Accuracy: 0.9647783251231526
k = 7 Accuracy: 0.9507389162561577
k = 9 Accuracy: 0.9514778325123153
k = 11 Accuracy: 0.9435960591133006
k = 13 Accuracy: 0.9576354679802955
k = 15 Accuracy: 0.9645320197044335
Best k: 5
Best accuracy: 0.9647783251231526


In [138]:
model = Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=best_k))])
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9444444444444444


In [152]:
from sklearn.model_selection import GridSearchCV

model = KNeighborsClassifier()
param_grid = {'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15]}

grid_search = GridSearchCV(model, param_grid, cv=splitter, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best val accuracy:", grid_search.best_score_)

y_pred  = grid_search.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

Best parameters: {'n_neighbors': 3}
Best val accuracy: 0.9507389162561577
Test Accuracy: 0.9444444444444444


In [156]:
grid_search = GridSearchCV(model, param_grid, cv=splitter, scoring='accuracy')

scores = cross_validate(grid_search, X, y, cv=splitter, scoring='accuracy')
accuracy = scores['test_score']
print("Accuracy:", accuracy)

Accuracy: [0.52777778 0.75       0.61111111 0.82857143 0.74285714]


In [157]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
    ])

param_grid = {'model__n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15]}

grid_search = GridSearchCV(pipeline, param_grid, cv=splitter, scoring='accuracy')

scores = cross_validate(grid_search, X, y, cv=splitter, scoring='accuracy')
accuracy = scores['test_score']
print("Accuracy:", accuracy)

Accuracy: [0.97222222 1.         0.91666667 0.97142857 0.97142857]
